# 05 - Hybrid Feature Training And Evaluation

This notebook trains classifiers using:

1. handcrafted ECG features only,
2. EfficientNet deep features only,
3. handcrafted + EfficientNet hybrid features.

The goal is to show whether deep feature extraction improves over the original baseline.


In [ ]:
from pathlib import Path
import json
import shutil
import sys
import time
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except Exception:
    LIGHTGBM_AVAILABLE = False

PROJECT_DIR = Path("..").resolve()
SRC_DIR = PROJECT_DIR / "src"
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(SRC_DIR))

from artifacts import build_model_metadata, save_json
from config import CONFIG
from handcrafted_features_v2 import HANDCRAFTED_FEATURES
from reproducibility import set_global_seed

LABEL_MAP = {0: "Normal", 1: "Arrhythmia", 2: "Other / Unknown"}
MODEL_NAME = CONFIG.active_model_name
seed_state = set_global_seed(CONFIG.seed)

print("LightGBM available:", LIGHTGBM_AVAILABLE)
print("Feature directory:", CONFIG.feature_dir)
print("Seed state:", seed_state)


## Load Feature Tables

Run `04_efficientnet_embeddings.ipynb` first. It creates both the deep feature table and the hybrid table.


In [ ]:
deep_path = CONFIG.deep_feature_path(MODEL_NAME)
hybrid_path = CONFIG.hybrid_feature_path(MODEL_NAME)

deep_df = pd.read_csv(deep_path)
hybrid_df = pd.read_csv(hybrid_path)

print("Deep feature table:", deep_df.shape)
print("Hybrid feature table:", hybrid_df.shape)
display(hybrid_df.head())


In [ ]:
DROP_HANDCRAFTED = {"filename", "record_id", "label", "label_name", "label_encoded", "age", "sex"}
DROP_DEEP = {"record_id", "label", "label_name"}

deep_cols = [c for c in deep_df.columns if c.startswith("eff_")]
handcrafted_cols = [c for c in HANDCRAFTED_FEATURES if c in hybrid_df.columns]
hybrid_cols = handcrafted_cols + deep_cols

assert len(handcrafted_cols) == len(HANDCRAFTED_FEATURES), "Missing enhanced handcrafted features."
assert len(deep_cols) == CONFIG.model_embedding_dim(MODEL_NAME), "Unexpected deep embedding width."
print("Handcrafted feature count:", len(handcrafted_cols))
print("Deep feature count:", len(deep_cols))
print("Hybrid feature count:", len(hybrid_cols))


## Train/Test Split

All three experiments use the same records and same split so the comparison is fair.


In [ ]:
y = hybrid_df["label"].astype(int)
train_idx, test_idx = train_test_split(
    hybrid_df.index,
    test_size=CONFIG.test_size,
    random_state=CONFIG.seed,
    stratify=y,
)

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
display(y.value_counts().sort_index().rename(index=LABEL_MAP).rename("count").reset_index())


## Model Definitions

PCA is used for deep and hybrid features because CNN embeddings have many dimensions.


In [ ]:
def make_lgbm():
    if not LIGHTGBM_AVAILABLE:
        return None
    return LGBMClassifier(
        n_estimators=CONFIG.n_estimators,
        learning_rate=0.04,
        max_depth=7,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9,
        class_weight="balanced",
        random_state=CONFIG.seed,
        n_jobs=CONFIG.n_jobs,
        verbose=-1,
    )


def make_model(use_pca=False):
    classifier = make_lgbm()
    if classifier is None:
        classifier = LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=CONFIG.seed,
            n_jobs=CONFIG.n_jobs,
        )

    steps = [("scaler", StandardScaler())]
    if use_pca:
        steps.append(("pca", PCA(n_components=CONFIG.pca_components, random_state=CONFIG.seed)))
    steps.append(("classifier", classifier))
    return Pipeline(steps)


experiments = {
    "handcrafted_only": {
        "features": handcrafted_cols,
        "model": make_model(use_pca=False),
    },
    "efficientnet_only": {
        "features": deep_cols,
        "model": make_model(use_pca=True),
    },
    "hybrid_handcrafted_efficientnet": {
        "features": hybrid_cols,
        "model": make_model(use_pca=True),
    },
}


## Cross-Validation And Held-Out Test Evaluation


In [ ]:
cv = StratifiedKFold(n_splits=CONFIG.cv_folds, shuffle=True, random_state=CONFIG.seed)
summary_rows = []
trained_models = {}

for name, config in experiments.items():
    feature_cols = config["features"]
    model = config["model"]
    X = hybrid_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    started = time.perf_counter()
    cv_results = cross_validate(
        model,
        X.iloc[train_idx],
        y.iloc[train_idx],
        cv=cv,
        scoring=["accuracy", "f1_macro", "f1_weighted"],
        n_jobs=CONFIG.n_jobs,
    )

    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    pred = model.predict(X.iloc[test_idx])

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X.iloc[test_idx])
        try:
            auc_ovr = roc_auc_score(y.iloc[test_idx], proba, multi_class="ovr", average="weighted")
        except Exception:
            auc_ovr = np.nan
    else:
        auc_ovr = np.nan

    trained_models[name] = {
        "model": model,
        "features": feature_cols,
        "pred": pred,
    }

    summary_rows.append({
        "experiment": name,
        "feature_count": len(feature_cols),
        "cv_accuracy_mean": cv_results["test_accuracy"].mean(),
        "cv_macro_f1_mean": cv_results["test_f1_macro"].mean(),
        "cv_weighted_f1_mean": cv_results["test_f1_weighted"].mean(),
        "test_accuracy": accuracy_score(y.iloc[test_idx], pred),
        "test_macro_f1": f1_score(y.iloc[test_idx], pred, average="macro"),
        "test_weighted_f1": f1_score(y.iloc[test_idx], pred, average="weighted"),
        "test_auc_ovr_weighted": auc_ovr,
        "runtime_seconds": time.perf_counter() - started,
    })

results = pd.DataFrame(summary_rows).sort_values("test_macro_f1", ascending=False).reset_index(drop=True)
results_path = CONFIG.result_dir / f"hybrid_comparison_{MODEL_NAME}.csv"
results.to_csv(results_path, index=False)

display(results)
print(f"Saved comparison: {results_path}")


In [ ]:
plt.figure(figsize=(10, 4.8))
plot_df = results.melt(
    id_vars=["experiment"],
    value_vars=["test_accuracy", "test_macro_f1", "test_weighted_f1"],
    var_name="metric",
    value_name="score",
)
sns.barplot(data=plot_df, x="score", y="experiment", hue="metric")
plt.axvline(0.80, color="red", linestyle="--", label="0.80 target")
plt.title("Held-Out Test Performance")
plt.xlim(0, 1)
plt.tight_layout()
plt.savefig(CONFIG.result_dir / f"hybrid_comparison_{MODEL_NAME}.png", dpi=160)
plt.show()


## Best Model Details


In [ ]:
best_name = results.iloc[0]["experiment"]
best = trained_models[best_name]
best_pred = best["pred"]

report = classification_report(
    y.iloc[test_idx],
    best_pred,
    target_names=[LABEL_MAP[i] for i in sorted(y.unique())],
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).T
report_path = CONFIG.result_dir / f"best_model_classification_report_{MODEL_NAME}.csv"
report_df.to_csv(report_path)

cm = confusion_matrix(y.iloc[test_idx], best_pred, labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(cm, display_labels=[LABEL_MAP[i] for i in sorted(y.unique())])
disp.plot(cmap="Blues", values_format="d")
plt.title(f"Best Model Confusion Matrix: {best_name}")
plt.tight_layout()
plt.savefig(CONFIG.result_dir / f"best_model_confusion_matrix_{MODEL_NAME}.png", dpi=160)
plt.show()

display(report_df)
print("Best experiment:", best_name)
print(f"Saved report: {report_path}")


In [ ]:
def backup_if_exists(path):
    if path.exists():
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup = path.with_name(f"{path.stem}_{stamp}.bak{path.suffix}")
        shutil.copy2(path, backup)
        print(f"Backed up existing artifact: {backup}")


pipeline = best["model"]
scaler = pipeline.named_steps["scaler"]
pca = pipeline.named_steps.get("pca")
classifier = pipeline.named_steps["classifier"]

artifact_paths = [
    CONFIG.model_artifact_path(MODEL_NAME),
    CONFIG.feature_list_path(MODEL_NAME),
    CONFIG.scaler_path(MODEL_NAME),
    CONFIG.pca_path(MODEL_NAME),
    CONFIG.metadata_path(MODEL_NAME),
]
for path in artifact_paths:
    backup_if_exists(path)

joblib.dump(classifier, CONFIG.model_artifact_path(MODEL_NAME))
joblib.dump(best["features"], CONFIG.feature_list_path(MODEL_NAME))
joblib.dump(scaler, CONFIG.scaler_path(MODEL_NAME))
joblib.dump(pca, CONFIG.pca_path(MODEL_NAME))

best_metrics = results.iloc[0].to_dict()
per_class_recall = {
    name: float(report[name]["recall"])
    for name in [LABEL_MAP[i] for i in sorted(y.unique())]
}
metadata = build_model_metadata(
    model_name=MODEL_NAME,
    dataset_size=len(hybrid_df),
    features=best["features"],
    metrics={**best_metrics, "per_class_recall": per_class_recall},
    seed=CONFIG.seed,
    extra={
        "best_experiment": best_name,
        "embedding_dimension": CONFIG.model_embedding_dim(MODEL_NAME),
        "target_size": list(CONFIG.model_target_size(MODEL_NAME)),
        "split_method": "record-level stratified train_test_split",
    },
)
save_json(metadata, CONFIG.metadata_path(MODEL_NAME))

print("Saved versioned classifier, feature list, scaler, PCA, and metadata.")
